# Treinamento de Detector de Deepfake v2 - MELHORADO

Este notebook treina modelos EfficientNet-B0, EfficientNet-B3, EfficientNet-B4 e AlexNet para detecção de deepfakes e faz um comparativo entre eles.

## Melhorias em relação à v1:
- **Focal Loss** - lida melhor com classes desbalanceadas
- **Class Weights** - compensa desbalanceamento de dados
- **Augmentations agressivas** - mais variação nos fakes
- **Label Smoothing** - regularização para melhor generalização
- **Threshold Calibration** - encontra o threshold ótimo
- **Warmup + Cosine Annealing** - scheduler mais robusto
- **Mais épocas (20)** - tempo suficiente para convergir
- **Mixup augmentation** - regularização adicional

**Ambiente:** Google Colab Pro (GPU A100/V100)


## 1. Setup do Ambiente

In [ ]:
# Verificar GPU
!nvidia-smi

# Instalar dependências
!pip install -q timm albumentations kaggle scikit-learn matplotlib seaborn

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    confusion_matrix, precision_recall_curve, f1_score
)
import matplotlib.pyplot as plt
import seaborn as sns

# Reprodutibilidade
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memória: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Download do Dataset

In [ ]:
# Configurar Kaggle (faça upload do kaggle.json)
from google.colab import files

# Descomente para fazer upload:
# uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/ 2>/dev/null || echo 'Faça upload do kaggle.json'
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Download dataset 140k Real and Fake Faces
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p /content/data --unzip

## 3. Configurações MELHORADAS

In [ ]:
class Config:
    # Paths
    DATA_ROOT = Path('/content/data/real_vs_fake/real-vs-fake')
    OUTPUT_DIR = Path('/content/output')
    
    # Modelos para comparativo
    MODEL_NAMES = ['efficientnet_b0', 'efficientnet_b3', 'efficientnet_b4', 'alexnet']
    PRIMARY_MODEL_NAME = 'efficientnet_b0'
    PRETRAINED = True
    NUM_CLASSES = 1
    
    # Training - MELHORADO
    EPOCHS = 20           # Mais épocas
    BATCH_SIZE = 48       # Batch menor para mais updates
    LEARNING_RATE = 3e-4  # LR um pouco maior
    WEIGHT_DECAY = 1e-4   # Mais regularização
    WARMUP_EPOCHS = 2     # Warmup do LR
    
    # Loss
    USE_FOCAL_LOSS = True   # Focal Loss em vez de BCE
    FOCAL_GAMMA = 2.0       # Gamma para Focal Loss
    LABEL_SMOOTHING = 0.1   # Label smoothing
    
    # Augmentation
    USE_MIXUP = True
    MIXUP_ALPHA = 0.4
    
    # Image
    IMG_SIZE = 224
    
    # Training
    USE_AMP = True
    NUM_WORKERS = 4
    PATIENCE = 5
    
    # Threshold
    CALIBRATE_THRESHOLD = True


    # Leakage check
    LEAKAGE_CHECK = True
    LEAKAGE_HASH_MAX_FILES = None  # Use int (ex: 20000) para auditoria mais rápida
    LEAKAGE_CHECK_DHASH = True
    LEAKAGE_SEED = 42

config = Config()
config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 4. Focal Loss (lida melhor com desbalanceamento)

In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss para classificação binária.
    Dá mais peso aos exemplos difíceis de classificar.
    """
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha  # Class weight
        self.label_smoothing = label_smoothing
    
    def forward(self, logits, targets):
        # Label smoothing
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + 0.5 * self.label_smoothing
        
        # BCE
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        
        # Probabilities
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        
        # Focal weight
        focal_weight = (1 - p_t) ** self.gamma
        
        # Alpha weight (class balancing)
        if self.alpha is not None:
            alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
            focal_weight = alpha_t * focal_weight
        
        focal_loss = focal_weight * bce_loss
        
        return focal_loss.mean()


class LabelSmoothingBCE(nn.Module):
    """BCE com Label Smoothing."""
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
    
    def forward(self, logits, targets):
        targets = targets * (1 - self.smoothing) + 0.5 * self.smoothing
        return F.binary_cross_entropy_with_logits(logits, targets)

## 5. Augmentations AGRESSIVAS

In [ ]:
def get_transforms(is_train=True):
    """Augmentations mais agressivas para treino."""
    if is_train:
        return A.Compose([
            A.Resize(config.IMG_SIZE, config.IMG_SIZE),
            
            # Flip e rotação
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.15, 
                scale_limit=0.2, 
                rotate_limit=20, 
                border_mode=0,
                p=0.5
            ),
            
            # Cor - MAIS AGRESSIVO
            A.OneOf([
                A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=1),
                A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=1),
                A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=1),
            ], p=0.7),
            
            # Ruído e blur - simula compressão
            A.OneOf([
                A.GaussNoise(var_limit=(10.0, 80.0), p=1),
                A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1),
                A.MultiplicativeNoise(multiplier=(0.9, 1.1), p=1),
            ], p=0.4),
            
            A.OneOf([
                A.GaussianBlur(blur_limit=(3, 7), p=1),
                A.MotionBlur(blur_limit=5, p=1),
                A.MedianBlur(blur_limit=5, p=1),
            ], p=0.3),
            
            # Compressão JPEG - IMPORTANTE para deepfakes
            A.ImageCompression(quality_lower=40, quality_upper=95, p=0.5),
            
            # Dropout - simula oclusões
            A.OneOf([
                A.CoarseDropout(max_holes=8, max_height=24, max_width=24, 
                               min_holes=2, min_height=8, min_width=8, p=1),
                A.GridDropout(ratio=0.2, p=1),
            ], p=0.3),
            
            # Normalize
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(config.IMG_SIZE, config.IMG_SIZE),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])

## 6. Mixup Augmentation

In [ ]:
def mixup_data(x, y, alpha=0.4):
    """
    Mixup: mistura duas imagens e seus labels.
    Ajuda o modelo a generalizar melhor.
    """
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Loss para Mixup."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

## 7. Dataset com Balanceamento

In [ ]:
class DeepfakeDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        try:
            image = np.array(Image.open(img_path).convert('RGB'))
        except:
            # Fallback to random noise if image fails
            image = np.random.randint(0, 255, (config.IMG_SIZE, config.IMG_SIZE, 3), dtype=np.uint8)
        
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']
        
        return image, torch.tensor(label, dtype=torch.float32)


def prepare_data(data_root):
    """Prepara dados com análise de balanceamento."""
    data = {'train': {'paths': [], 'labels': []},
            'valid': {'paths': [], 'labels': []},
            'test': {'paths': [], 'labels': []}}
    
    for split in ['train', 'valid', 'test']:
        split_path = data_root / split
        
        # Real (label = 0)
        real_path = split_path / 'real'
        if real_path.exists():
            for img in real_path.glob('*.jpg'):
                data[split]['paths'].append(str(img))
                data[split]['labels'].append(0)
        
        # Fake (label = 1)
        fake_path = split_path / 'fake'
        if fake_path.exists():
            for img in fake_path.glob('*.jpg'):
                data[split]['paths'].append(str(img))
                data[split]['labels'].append(1)
        
        # Estatísticas
        labels = data[split]['labels']
        n_real = labels.count(0)
        n_fake = labels.count(1)
        total = len(labels)
        print(f'{split}: {total} imgs | Real: {n_real} ({100*n_real/total:.1f}%) | Fake: {n_fake} ({100*n_fake/total:.1f}%)')
    
    return data

# Preparar dados
print('📊 Análise do Dataset:')
data = prepare_data(config.DATA_ROOT)

## 7.1. Checagem Anti-Leakage

Verifica vazamento entre `train/valid/test` por path, hash exato (MD5) e hash perceptual (dHash).


In [ ]:
# Auditoria anti-leakage entre splits
import hashlib
from itertools import combinations


def file_md5(path, chunk_size=1 << 20):
    """Hash exato do arquivo (detecta duplicata byte a byte)."""
    h = hashlib.md5()
    try:
        with open(path, 'rb') as f:
            while True:
                chunk = f.read(chunk_size)
                if not chunk:
                    break
                h.update(chunk)
        return h.hexdigest()
    except Exception:
        return None


def dhash(path, hash_size=8):
    """dHash perceptual simples (detecta quase-duplicatas visuais)."""
    try:
        img = Image.open(path).convert('L').resize((hash_size + 1, hash_size))
        arr = np.array(img)
        diff = arr[:, 1:] > arr[:, :-1]
        bits = ''.join('1' if v else '0' for v in diff.flatten())
        return f'{int(bits, 2):0{hash_size * hash_size // 4}x}'
    except Exception:
        return None


def build_hash_set(paths, hash_fn, max_files=None, seed=42, desc='Hash'):
    if max_files is not None and max_files < len(paths):
        rng = random.Random(seed)
        selected = rng.sample(paths, max_files)
    else:
        selected = paths

    hashes = []
    for p in tqdm(selected, desc=desc):
        h = hash_fn(p)
        if h is not None:
            hashes.append(h)

    return set(hashes), len(selected), len(hashes)


if config.LEAKAGE_CHECK:
    split_paths = {k: data[k]['paths'] for k in ['train', 'valid', 'test']}
    split_sets = {k: set(v) for k, v in split_paths.items()}

    print('\n🔎 Checagem anti-leakage entre splits')
    print('=' * 70)

    # 1) Overlap direto por caminho
    print('\n1) Overlap por caminho (path):')
    for a, b in combinations(['train', 'valid', 'test'], 2):
        overlap = split_sets[a].intersection(split_sets[b])
        print(f'  {a} x {b}: {len(overlap)}')
        if overlap:
            print(f'    Exemplo: {next(iter(overlap))}')

    # 2) Overlap por hash exato (MD5)
    max_files = config.LEAKAGE_HASH_MAX_FILES
    if max_files is None:
        print('\n2) Overlap por MD5 (full dataset):')
    else:
        print(f'\n2) Overlap por MD5 (amostra por split = {max_files}):')

    md5_sets = {}
    for split in ['train', 'valid', 'test']:
        hs, n_sel, n_ok = build_hash_set(
            split_paths[split],
            file_md5,
            max_files=max_files,
            seed=config.LEAKAGE_SEED,
            desc=f'MD5 {split}'
        )
        md5_sets[split] = hs
        print(f'  {split}: arquivos analisados={n_sel}, hashes válidos={n_ok}, únicos={len(hs)}')

    md5_any_overlap = False
    for a, b in combinations(['train', 'valid', 'test'], 2):
        overlap = md5_sets[a].intersection(md5_sets[b])
        print(f'  {a} x {b}: {len(overlap)} hashes iguais')
        if overlap:
            md5_any_overlap = True

    # 3) Overlap por hash perceptual (dHash)
    dhash_any_overlap = False
    if config.LEAKAGE_CHECK_DHASH:
        if max_files is None:
            print('\n3) Overlap por dHash (full dataset):')
        else:
            print(f'\n3) Overlap por dHash (amostra por split = {max_files}):')

        dhash_sets = {}
        for split in ['train', 'valid', 'test']:
            hs, n_sel, n_ok = build_hash_set(
                split_paths[split],
                dhash,
                max_files=max_files,
                seed=config.LEAKAGE_SEED,
                desc=f'dHash {split}'
            )
            dhash_sets[split] = hs
            print(f'  {split}: arquivos analisados={n_sel}, hashes válidos={n_ok}, únicos={len(hs)}')

        for a, b in combinations(['train', 'valid', 'test'], 2):
            overlap = dhash_sets[a].intersection(dhash_sets[b])
            print(f'  {a} x {b}: {len(overlap)} hashes perceptuais iguais')
            if overlap:
                dhash_any_overlap = True

    # Resumo
    path_overlap = any(len(split_sets[a].intersection(split_sets[b])) > 0 for a, b in combinations(['train', 'valid', 'test'], 2))

    print('\n' + '=' * 70)
    if not path_overlap and not md5_any_overlap and not dhash_any_overlap:
        print('✅ Sem indícios de leakage entre train/valid/test nas checagens aplicadas.')
    else:
        print('⚠️ Possível leakage detectado. Revise os pares com overlap acima.')
        print('   Dica: comece pelos hashes MD5 iguais (duplicata exata).')
else:
    print('Checagem anti-leakage desativada em config.LEAKAGE_CHECK')


In [ ]:
# Calcular class weights para balanceamento
train_labels = data['train']['labels']
n_real = train_labels.count(0)
n_fake = train_labels.count(1)
total = len(train_labels)

# Weight inversamente proporcional à frequência
weight_real = total / (2 * n_real)
weight_fake = total / (2 * n_fake)

# Para Focal Loss - alpha é o peso de positivos (fake)
focal_alpha = n_real / total  # Se há mais reais, dar mais peso aos fakes

print(f'\n⚖️ Class Weights:')
print(f'Real (0): {weight_real:.3f}')
print(f'Fake (1): {weight_fake:.3f}')
print(f'Focal Alpha: {focal_alpha:.3f}')

In [ ]:
# Criar datasets
train_dataset = DeepfakeDataset(
    data['train']['paths'],
    data['train']['labels'],
    transform=get_transforms(is_train=True)
)

valid_dataset = DeepfakeDataset(
    data['valid']['paths'],
    data['valid']['labels'],
    transform=get_transforms(is_train=False)
)

test_dataset = DeepfakeDataset(
    data['test']['paths'],
    data['test']['labels'],
    transform=get_transforms(is_train=False)
)

# WeightedRandomSampler para balancear batches
sample_weights = [weight_fake if l == 1 else weight_real for l in train_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    sampler=sampler,  # Usa sampler balanceado
    num_workers=config.NUM_WORKERS,
    pin_memory=True,
    drop_last=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=True
)

print(f'\n📦 DataLoaders:')
print(f'Train: {len(train_loader)} batches')
print(f'Valid: {len(valid_loader)} batches')
print(f'Test: {len(test_loader)} batches')

## 8. Modelo com Regularização Extra

In [ ]:
class DeepfakeDetector(nn.Module):
    def __init__(self, model_name='efficientnet_b0', pretrained=True, num_classes=1, dropout=0.4):
        super().__init__()
        
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0
        )
        
        self.num_features = self.backbone.num_features
        
        # Classifier - MESMA arquitetura do worker para compatibilidade
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.num_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        logits = self.classifier(features)
        return logits.squeeze(-1)


def build_model(model_name):
    return DeepfakeDetector(
        model_name=model_name,
        pretrained=config.PRETRAINED,
        num_classes=config.NUM_CLASSES,
        dropout=0.4
    ).to(device)


# Modelo inicial para inspeção
model = build_model(config.PRIMARY_MODEL_NAME)

print(f'Modelos no comparativo: {config.MODEL_NAMES}')
print(f'Modelo inicial: {config.PRIMARY_MODEL_NAME}')
print(f'Parâmetros ({config.PRIMARY_MODEL_NAME}): {sum(p.numel() for p in model.parameters()):,}')


## 9. Warmup Scheduler

In [ ]:
class WarmupCosineScheduler:
    """Learning rate scheduler com warmup linear e decaimento cosseno."""
    def __init__(self, optimizer, warmup_epochs, total_epochs, base_lr, min_lr=1e-6):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.current_epoch = 0
    
    def step(self):
        self.current_epoch += 1
        
        if self.current_epoch <= self.warmup_epochs:
            # Linear warmup
            lr = self.base_lr * (self.current_epoch / self.warmup_epochs)
        else:
            # Cosine decay
            progress = (self.current_epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + np.cos(np.pi * progress))
        
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
        
        return lr
    
    def get_lr(self):
        return self.optimizer.param_groups[0]['lr']

## 10. Training Loop MELHORADO

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler, device, use_mixup=True):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    pbar = tqdm(loader, desc='Training')
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        # Mixup
        if use_mixup and config.USE_MIXUP and np.random.rand() < 0.5:
            images, labels_a, labels_b, lam = mixup_data(images, labels, config.MIXUP_ALPHA)
            
            with torch.amp.autocast('cuda', enabled=config.USE_AMP):
                outputs = model(images)
                loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
            
            # Use original labels for metrics
            labels_for_metrics = labels_a
        else:
            with torch.amp.autocast('cuda', enabled=config.USE_AMP):
                outputs = model(images)
                loss = criterion(outputs, labels)
            labels_for_metrics = labels
        
        scaler.scale(loss).backward()
        
        # Gradient clipping
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
        
        preds = torch.sigmoid(outputs).detach().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels_for_metrics.cpu().numpy())
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_preds = np.array(all_preds)
    epoch_labels = np.array(all_labels)
    
    # Métricas
    accuracy = accuracy_score(epoch_labels > 0.5, epoch_preds > 0.5)
    try:
        auc = roc_auc_score(epoch_labels > 0.5, epoch_preds)
    except:
        auc = 0.5
    
    return epoch_loss, accuracy, auc


def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        pbar = tqdm(loader, desc='Validation')
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)
            
            with torch.amp.autocast('cuda', enabled=config.USE_AMP):
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            
            preds = torch.sigmoid(outputs).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_preds = np.array(all_preds)
    epoch_labels = np.array(all_labels)
    
    accuracy = accuracy_score(epoch_labels, epoch_preds > 0.5)
    auc = roc_auc_score(epoch_labels, epoch_preds)
    
    return epoch_loss, accuracy, auc, epoch_preds, epoch_labels

In [ ]:
# Setup training
if config.USE_FOCAL_LOSS:
    criterion = FocalLoss(gamma=config.FOCAL_GAMMA, alpha=focal_alpha, label_smoothing=config.LABEL_SMOOTHING)
    print('📍 Usando Focal Loss')
else:
    criterion = LabelSmoothingBCE(smoothing=config.LABEL_SMOOTHING)
    print('📍 Usando BCE com Label Smoothing')

model_histories = {}
model_summaries = []
best_model_name = None
best_model_auc = -1.0


In [ ]:
# Training loop (comparativo entre arquiteturas)
for model_name in config.MODEL_NAMES:
    print('\n' + '=' * 70)
    print(f'🚀 Treinando modelo: {model_name}')
    print('=' * 70)
    
    model = build_model(model_name)
    optimizer = optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
    scheduler = WarmupCosineScheduler(
        optimizer,
        warmup_epochs=config.WARMUP_EPOCHS,
        total_epochs=config.EPOCHS,
        base_lr=config.LEARNING_RATE
    )
    scaler = GradScaler(enabled=config.USE_AMP)
    
    history = {
        'train_loss': [], 'train_acc': [], 'train_auc': [],
        'valid_loss': [], 'valid_acc': [], 'valid_auc': [],
        'lr': []
    }
    
    best_auc = 0.0
    best_epoch = 0
    patience_counter = 0
    
    print(f'Épocas: {config.EPOCHS} | Batch: {config.BATCH_SIZE} | LR: {config.LEARNING_RATE}')
    
    for epoch in range(config.EPOCHS):
        current_lr = scheduler.step()
        print(f'\nÉpoca {epoch + 1}/{config.EPOCHS} | LR: {current_lr:.2e}')
        print('-' * 50)
        
        # Train
        train_loss, train_acc, train_auc = train_epoch(
            model, train_loader, criterion, optimizer, scaler, device,
            use_mixup=(epoch > config.WARMUP_EPOCHS)  # Mixup após warmup
        )
        
        # Validate
        valid_loss, valid_acc, valid_auc, valid_preds, valid_labels = validate_epoch(
            model, valid_loader, criterion, device
        )
        
        # History
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['train_auc'].append(train_auc)
        history['valid_loss'].append(valid_loss)
        history['valid_acc'].append(valid_acc)
        history['valid_auc'].append(valid_auc)
        history['lr'].append(current_lr)
        
        # Distribuição das predições
        fake_preds = valid_preds[valid_labels == 1]
        real_preds = valid_preds[valid_labels == 0]
        
        print(f'Train - Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | AUC: {train_auc:.4f}')
        print(f'Valid - Loss: {valid_loss:.4f} | Acc: {valid_acc:.4f} | AUC: {valid_auc:.4f}')
        print(f'📊 Preds - Fake mean: {fake_preds.mean():.3f} | Real mean: {real_preds.mean():.3f}')
        
        # Save best por modelo
        if valid_auc > best_auc:
            best_auc = valid_auc
            best_epoch = epoch + 1
            patience_counter = 0
            
            checkpoint = {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_auc': best_auc,
                'valid_acc': valid_acc,
                'config': {
                    'model_name': model_name,
                    'img_size': config.IMG_SIZE,
                }
            }
            checkpoint_path = config.OUTPUT_DIR / f'best_model_{model_name}.pth'
            torch.save(checkpoint, checkpoint_path)
            print(f'✅ Melhor checkpoint ({model_name}) salvo! AUC: {best_auc:.4f}')
        else:
            patience_counter += 1
            if patience_counter >= config.PATIENCE:
                print(f'\n⏹️ Early stopping para {model_name} após {epoch + 1} épocas')
                break
    
    model_histories[model_name] = history
    model_summaries.append({
        'model_name': model_name,
        'best_valid_auc': best_auc,
        'best_epoch': best_epoch,
        'params': sum(p.numel() for p in model.parameters())
    })
    
    if best_auc > best_model_auc:
        best_model_auc = best_auc
        best_model_name = model_name

comparison_df = pd.DataFrame(model_summaries).sort_values('best_valid_auc', ascending=False).reset_index(drop=True)

print('\n' + '=' * 70)
print('📌 Comparativo entre modelos (validation):')
print(comparison_df)
print(f'\n🏆 Melhor modelo no validation: {best_model_name} (AUC: {best_model_auc:.4f})')


## 11. Threshold Calibration

In [ ]:
def find_best_threshold(labels, preds):
    """Encontra o threshold que maximiza F1-score."""
    thresholds = np.arange(0.1, 0.9, 0.01)
    best_f1 = 0
    best_threshold = 0.5
    
    results = []
    for threshold in thresholds:
        preds_binary = (preds > threshold).astype(int)
        f1 = f1_score(labels, preds_binary)
        acc = accuracy_score(labels, preds_binary)
        results.append({'threshold': threshold, 'f1': f1, 'accuracy': acc})
        
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
    
    return best_threshold, best_f1, pd.DataFrame(results)


# Carregar melhor modelo do comparativo
best_checkpoint_path = config.OUTPUT_DIR / f'best_model_{best_model_name}.pth'
checkpoint = torch.load(best_checkpoint_path)

model = build_model(checkpoint['config']['model_name'])
model.load_state_dict(checkpoint['model_state_dict'])

print(f'Modelo carregado: {checkpoint["config"]["model_name"]} | epoch {checkpoint["epoch"]} | AUC {checkpoint["best_auc"]:.4f}')


# Avaliar no validation
_, _, _, valid_preds, valid_labels = validate_epoch(model, valid_loader, criterion, device)

# Encontrar melhor threshold
best_threshold, best_f1, threshold_df = find_best_threshold(valid_labels, valid_preds)
print(f'\n🎯 Melhor Threshold ({best_model_name}): {best_threshold:.2f} (F1: {best_f1:.4f})')

# Plot
plt.figure(figsize=(10, 4))
plt.plot(threshold_df['threshold'], threshold_df['f1'], label='F1-Score')
plt.plot(threshold_df['threshold'], threshold_df['accuracy'], label='Accuracy')
plt.axvline(x=best_threshold, color='r', linestyle='--', label=f'Best: {best_threshold:.2f}')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title(f'Threshold Calibration - {best_model_name}')
plt.legend()
plt.grid(True)
plt.show()


## 12. Avaliação Final no Test Set

In [ ]:
# Avaliar no test set com o melhor modelo do comparativo
test_loss, test_acc, test_auc, test_preds, test_labels = validate_epoch(
    model, test_loader, criterion, device
)

# Com threshold otimizado
test_preds_binary = (test_preds > best_threshold).astype(int)
test_acc_calibrated = accuracy_score(test_labels, test_preds_binary)
test_f1_calibrated = f1_score(test_labels, test_preds_binary)

print('=' * 60)
print('📊 RESULTADOS FINAIS NO TEST SET')
print('=' * 60)
print(f'Modelo avaliado: {best_model_name}')
print(f'AUC-ROC: {test_auc:.4f}')
print(f'\nCom threshold = 0.5:')
print(f'  Accuracy: {test_acc:.4f}')
print(f'\nCom threshold otimizado = {best_threshold:.2f}:')
print(f'  Accuracy: {test_acc_calibrated:.4f}')
print(f'  F1-Score: {test_f1_calibrated:.4f}')

print('\n' + classification_report(test_labels, test_preds_binary, target_names=['Real', 'Fake']))


In [ ]:
# Confusion Matrix
cm = confusion_matrix(test_labels, test_preds_binary)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Real', 'Fake'],
            yticklabels=['Real', 'Fake'])
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title(f'Confusion Matrix - {best_model_name} (threshold={best_threshold:.2f})')
plt.show()

# Distribuição das predições
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(test_preds[test_labels == 0], bins=50, alpha=0.7, label='Real', color='green')
axes[0].hist(test_preds[test_labels == 1], bins=50, alpha=0.7, label='Fake', color='red')
axes[0].axvline(x=best_threshold, color='black', linestyle='--', label=f'Threshold: {best_threshold:.2f}')
axes[0].set_xlabel('Probabilidade de Fake')
axes[0].set_ylabel('Contagem')
axes[0].set_title(f'Distribuição das Predições - {best_model_name}')
axes[0].legend()

# Precision-Recall curve
precision, recall, _ = precision_recall_curve(test_labels, test_preds)
axes[1].plot(recall, precision)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].grid(True)

plt.tight_layout()
plt.show()


## 13. Exportar Modelo Final

In [ ]:
# Exportar modelo para produção
export_path = config.OUTPUT_DIR / f'{best_model_name}_deepfake_v2.pth'

export_checkpoint = {
    'model_state_dict': model.state_dict(),
    'best_threshold': best_threshold,
    'test_auc': test_auc,
    'test_f1': test_f1_calibrated,
    'comparison': comparison_df.to_dict(orient='records'),
    'config': {
        'model_name': best_model_name,
        'img_size': config.IMG_SIZE,
        'normalize_mean': [0.485, 0.456, 0.406],
        'normalize_std': [0.229, 0.224, 0.225],
    }
}

torch.save(export_checkpoint, export_path)
print(f'✅ Modelo exportado: {export_path.name}')

# Mostrar tamanho
import os
size_mb = os.path.getsize(export_path) / 1e6
print(f'📦 Tamanho: {size_mb:.1f} MB')


In [ ]:
# Download do modelo
from google.colab import files
files.download(str(export_path))


## 14. Resumo das Melhorias

### O que mudou em relação à v1:

1. **Focal Loss** - Dá mais importância aos exemplos difíceis
2. **WeightedRandomSampler** - Batches balanceados
3. **Augmentations agressivas** - Especialmente compressão JPEG
4. **Label Smoothing** - Regularização
5. **Mixup** - Mistura imagens para generalização
6. **Warmup + Cosine Annealing** - LR mais estável
7. **Threshold Calibration** - Encontra o melhor ponto de corte
8. **Mais épocas (20)** - Tempo para convergir
9. **Comparativo de arquiteturas (EfficientNet-B0/B3/B4 e AlexNet)** - Seleção automática do melhor modelo
10. **Gradient Clipping** - Evita explosão de gradientes

### Próximos passos se ainda não funcionar:
- Usar EfficientNet-B3 ou B4 (mais capacidade)
- Adicionar mais dados (outros datasets)
- Face-specific augmentations (landmarks)
- Attention mechanisms
- Ensemble de modelos
